# Demo 03 - Gaps and Islands

This demo will look at gaps in data. Looking at gaps in data is important because it shows us where we might be missing individual data points. If this is a set which should be complete, such as check numbers or invoice numbers, it might give you a telling indicator that there might be an issue.

## Missing Line Items

In our data, invoice number is the key value we use for tracking money. This number should never have any gaps in it.

To test this, we will run a query which looks for gaps in the data. Note that this uses syntax which Microsoft introduced in SQL Server 2012, so if you are using SQL Server 2008 R2 or earlier, you will need to use a different technique to solve the problem.

In [ ]:
WITH C AS
(
	SELECT
		li.LineItemID AS CurrentLineItemID,
		LEAD(li.LineItemID) OVER (ORDER BY li.LineItemID) AS NextLineItemID
	FROM dbo.LineItem li
)
SELECT
	CurrentLineItemID + 1 AS rangestart,
	NextLineItemID - 1 AS rangeend
FROM C
WHERE
	NextLineItemID - CurrentLineItemID > 1;

This tells us about specific ranges of missing invoices, which is a concern.  We'll want to bring this up with the forensic accountants and see if they can figure out why these are missing.

## Islands of Customer-Invoice-Bus Days

Missing data is very interest, but so are clusters of contiguous work on buses.  We can expect multiple work items on a bus occasionally, so that by itself won't be crazy.  But let's see how the data looks.

Because of the possibility of multiple work items per bus on a single day, we'll aggregate data by day first so that we get one row per day.

In [ ]:
WITH input AS
(
    SELECT
        li.LineItemDate,
        DATEDIFF(DAY, '2011-01-01', li.LineItemDate) AS col,
        li.BusID
    FROM dbo.LineItem li
    GROUP BY
        li.LineItemDate,
        li.BusID
),
islands AS
(
    SELECT
        LineItemDate,
        BusID,
        col,
        col - DENSE_RANK() OVER (PARTITION BY BusID ORDER BY col) AS iq
    FROM input
)
SELECT
    i.BusID,
    MIN(i.LineItemDate) AS MinDate,
    MAX(i.LineItemDate) AS MaxDate,
    DATEDIFF(DAY, MIN(i.LineItemDate), MAX(i.LineItemDate)) + 1 AS ContiguousDays
FROM islands i
GROUP BY
    i.BusID,
    i.iq
HAVING
    DATEDIFF(DAY, MIN(i.LineItemDate), MAX(i.LineItemDate)) + 1 >= 5
ORDER BY
    MinDate DESC;

Based on this information, it looks like we first started seeing at least one invoice on five contiguous days back in 2014, but the biggest cluster is 2019-2021.  Still, even so, it appears to pop up a couple of times per year.

## Islands of Vendor-Invoice Days

Now let's look at which vendors are most popular and what the largest clusters of days are for these vendors.  The flow is pretty much the same, except instead of grouping by bus, we group by vendor.

In [ ]:
WITH input AS
(
    SELECT
        li.LineItemDate,
        DATEDIFF(DAY, '2011-01-01', li.LineItemDate) AS col,
        li.VendorID
    FROM dbo.LineItem li
    GROUP BY
        li.LineItemDate,
        li.VendorID
),
islands AS
(
    SELECT
        LineItemDate,
        VendorID,
        col,
        col - DENSE_RANK() OVER (PARTITION BY VendorID ORDER BY col) AS iq
    FROM input
),
calculatedIslands AS
(
    SELECT
        i.VendorID,
        MIN(i.LineItemDate) AS MinDate,
        MAX(i.LineItemDate) AS MaxDate,
        DATEDIFF(DAY, MIN(i.LineItemDate), MAX(i.LineItemDate)) + 1 AS ContiguousDays
    FROM islands i
    GROUP BY
        i.VendorID,
        i.iq
)
SELECT
    ci.VendorID,
    MAX(ci.ContiguousDays) AS MaxContiguousDays
FROM calculatedIslands ci
GROUP BY
    ci.VendorID;

Well...if you saw this in a real-life data set, you'd figure that something is probably wrong.

Breaking from character a little bit, this is an artifact of the data creation process rather than an actual problem.